# Simulacion de autovalores (eigenmode) con SQDMetal + Palace

Encuentra los **modos resonantes** de una estructura: su frecuencia y su
factor de calidad interno `Q` (limitado aca por la tangente de perdidas del
sustrato y las participaciones de energia en las interfaces).

Aca simulamos el **resonador solo**, sin la linea de transmision: eso da la
frecuencia y el `Q` intrinsecos. (Para la respuesta S21 medida, con la
linea acoplada, esta `build_and_driven.ipynb`.)

**Necesitas:** `qiskit-metal`, `SQDMetal`, y un binario `palace` compilado.


## 0. Parametros

In [ ]:
import os, time
import numpy as np
import pandas as pd

PALACE_BIN = os.environ.get("PALACE_BIN", "palace")   # <-- ruta a tu binario de palace
OUTDIR     = "."
NUM_CPUS   = 8                                        # nucleos (nproc en la terminal)

# geometria del resonador (cuarto de onda: extremo abierto -> meandro -> extremo a tierra)
TOTAL_LENGTH_MM = 7.48345
CPW_WIDTH_UM    = 10
CPW_GAP_UM      = 6

# solver de autovalores
# El shift-invert necesita que STARTING_FREQ este CERCA (idealmente un poco por
# debajo) del modo: si esta lejos, el operador desplazado (K - sigma^2 M) queda
# muy indefinido y el FGMRES + multigrilla de cada iteracion NO converge
# (residual lineal estancado -> el eigensolver nunca avanza).
# lambda/4 de 7.48 mm de CPW sobre silicio (eps_eff ~ 6.4) ~ 3.95 GHz; la capacidad
# del extremo abierto y el acople entre brazos del meandro lo bajan un poco.
# (Para una resonancia de 5 GHz el trazo tendria que medir ~5.9 mm.)
STARTING_FREQ_HZ = 3.6e9
NUMBER_OF_FREQS  = 3       # cuantos modos resolver contando desde el shift hacia arriba
SOLVER_ORDER     = 1       # orden 1 = ~320k incognitas y robusto; orden 2 = ~2M y menos estable. Subilo despues si el Q lo necesita
SOLVER_TOL       = 1e-8
SOLVER_MAXITS    = 500     # iteraciones del solver LINEAL por paso del eigensolver
SOLNS_TO_SAVE    = 1       # cuantos modos guardar el campo (para ParaView). 0 = ninguno

# mallado (tamano minimo de elemento, um). Pauta: ~la mitad de la brecha CPW (6um -> ~3um)
RES_MIN_SIZE_UM = 3.0
BG_MAX_SIZE_UM  = 200.0

## 1. Construir la geometria (resonador solo)

Extremo abierto (`OpenToGround`) -> meandro (`RouteMeander`, `total_length`
fija) -> extremo a tierra (`ShortToGround`). Sin launchpads ni linea.

El chip se hace **ajustado** al resonador: sin linea no hace falta un chip
ancho, y un dominio mas chico = menos elementos y empuja los modos de caja
por encima del resonador.

`design.rebuild()` fuerza a qiskit-metal a calcular los poligonos de cada
componente en `design.qgeometry.tables` -- SQDMetal lee esas tablas.


In [ ]:
from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround

design = designs.DesignPlanar({}, overwrite_enabled=True)
design.chips.main.size.size_x = '3.0mm'
design.chips.main.size.size_y = '2.4mm'
design.chips.main.size.size_z = '500um'
design.chips.main.size.center_x = '-0.2mm'
design.chips.main.size.center_y = '-0.65mm'
design.variables['cpw_width'] = f'{CPW_WIDTH_UM} um'
design.variables['cpw_gap']   = f'{CPW_GAP_UM} um'

OpenToGround(design, 'otg1', options=dict(chip='main', pos_x='-0.2mm', pos_y='-40um', orientation=180))
ShortToGround(design, 'stg2', options=dict(chip='main', pos_x='0mm', pos_y='-1.25mm', orientation=0))
RouteMeander(design, 'resonator1', Dict(
    trace_width=f'{CPW_WIDTH_UM}um', trace_gap=f'{CPW_GAP_UM}um',
    total_length=f'{TOTAL_LENGTH_MM}mm', hfss_wire_bonds=False, fillet='99.9 um',
    lead=dict(start_straight='10um'),
    meander=Dict(spacing="200um", assymetry="1mm"),
    pin_inputs=Dict(start_pin=Dict(component='otg1', pin='open'),
                    end_pin=Dict(component='stg2', pin='short'))))
design.rebuild()

# estimacion analitica (cuarto de onda, CPW sobre silicio, eps_eff ~ 6.45)
f_guess = 3e8 / np.sqrt(6.45) / (4 * TOTAL_LENGTH_MM * 1e-3)
print("componentes:", list(design.components.keys()))
print(f"estimacion lambda/4:  f ~ {f_guess/1e9:.3f} GHz")


In [ ]:
import matplotlib.pyplot as plt

def plot_design(design, ax=None, figsize=(8, 6)):
    """Vista estatica del diseno desde qgeometry.tables (mm). Headless-safe."""
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)
    for tname, table in design.qgeometry.tables.items():
        for _, row in table.iterrows():
            try:
                geom = row["geometry"]; sub = bool(row.get("subtract", False))
                if geom.geom_type == "LineString":
                    xs, ys = geom.xy
                    ax.plot([x*1e3 for x in xs], [y*1e3 for y in ys], color="#1a887b", lw=1.4)
                    continue
                for pp in (geom.geoms if geom.geom_type.startswith("Multi") else [geom]):
                    xs, ys = pp.exterior.xy
                    ax.fill([x*1e3 for x in xs], [y*1e3 for y in ys],
                            facecolor="none" if sub else "#b9791f",
                            edgecolor="#bd524a" if sub else "#555", alpha=0.55, lw=0.8)
            except Exception:
                pass
    ax.set_aspect("equal"); ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")
    ax.set_title("geometria (naranja = metal, verde = ruteo, rojo = corte en el plano de tierra)")
    return ax

plot_design(design);


## 2. Crear la simulacion de autovalores

`PALACE_Eigenmode_Simulation` convierte el diseno de qiskit-metal en un modelo
Palace. Opciones importantes:

| clave | significado |
|---|---|
| `starting_freq` / `number_of_freqs` | objetivo del shift-invert y cuantos modos devolver |
| `solns_to_save` | cuantos modos guardar el campo para ParaView |
| `solver_order` | orden polinomico de los elementos finitos (1 o 2) |
| `dielectric_material` | sustrato (`"silicon"`) |
| `palace_dir` / `num_cpus` | binario de palace y nucleos MPI |

`mode="simPC"` = corre Palace en esta maquina. `meshing="GMSH"` = mallador GMSH.


In [ ]:
from SQDMetal.PALACE.Eigenmode_Simulation import PALACE_Eigenmode_Simulation
from SQDMetal.Utilities.Materials import MaterialInterface

eigen_sim = PALACE_Eigenmode_Simulation(
    name="eig_resonator",
    metal_design=design,
    sim_parent_directory=OUTDIR,
    mode="simPC",
    meshing="GMSH",
    create_files=True,
    user_options={
        "mesh_refinement": 0,
        "dielectric_material": "silicon",
        "starting_freq": STARTING_FREQ_HZ,
        "number_of_freqs": NUMBER_OF_FREQS,
        "solns_to_save": SOLNS_TO_SAVE,
        "solver_order": SOLVER_ORDER,
        "solver_tol": SOLVER_TOL,
        "solver_maxits": SOLVER_MAXITS,
        "fillet_resolution": 12,
        "palace_dir": PALACE_BIN,
        "num_cpus": NUM_CPUS,
    })

## 3. Metal, plano de tierra -- y NADA de puertos

- `add_metallic(1)` -- la capa 1 de qiskit-metal es el metal (superconductor / PEC).
- `add_ground_plane()` -- rellena el plano de tierra menos cada brecha/corte del diseno.

En un eigenmode del resonador **aislado** NO se ponen puertos: un puerto de
50 ohm cargaria el modo y lo convertiria en un problema driveado. El eigenmode
te da el modo desnudo y su `Q` interno.


In [ ]:
eigen_sim.add_metallic(1)
eigen_sim.add_ground_plane()


## 4. Mallado local + interfaces EPR

- `fine_mesh_components(["resonator1"], ...)` -- afina el mallado cerca del
  resonador; `taper_dist_min/max` controlan que tan rapido vuelve al fondo.
- `setup_EPR_interfaces(...)` -- etiqueta las interfaces metal-aire /
  sustrato-aire / sustrato-metal para que Palace reporte la participacion de
  energia en cada una (la entrada para un presupuesto de perdidas / `1/Q`).


In [ ]:
eigen_sim.fine_mesh_components(
    ["resonator1"],
    min_size=RES_MIN_SIZE_UM * 1e-6, max_size=BG_MAX_SIZE_UM * 1e-6,
    taper_dist_min=3 * RES_MIN_SIZE_UM * 1e-6, taper_dist_max=10 * RES_MIN_SIZE_UM * 1e-6,
    metals_only=False)

eigen_sim.setup_EPR_interfaces(
    metal_air=MaterialInterface("Aluminium-Vacuum"),
    substrate_air=MaterialInterface("Silicon-Vacuum"),
    substrate_metal=MaterialInterface("Silicon-Aluminium"))


## 5. Preparar y correr

- `prepare_simulation()` -- corre GMSH y escribe el mallado + el JSON de Palace.
  Despues de ejecutar, mira `eig_resonator/.../eig_resonator.png` y el `.msh`
  para chequear que el mallado tenga sentido.
- `run()` -- invoca Palace. **Es el paso lento** (segundos a minutos).


In [ ]:
t = time.time()
eigen_sim.prepare_simulation()
print(f"mallado + preparacion: {time.time()-t:.1f} s")


In [ ]:
t = time.time()
eigen_sim.run()
print(f"solve: {time.time()-t:.1f} s")


## 6. Leer los resultados

Palace escribe `eig.csv`: una fila por modo convergido, con la frecuencia
compleja (parte real = resonancia, parte imaginaria fija el `Q`) y el `Q`.

**Ojo:** identifica el modo del resonador por el patron de campo / la
participacion, NO por el orden de frecuencia -- un modo espurio de caja del
dominio finito puede aparecer entre armonicos reales.


In [ ]:
df = pd.read_csv(eigen_sim._output_data_dir + "/eig.csv")
df.columns = df.columns.str.strip()
fcol = [c for c in df.columns if c.startswith("Re{f}")][0]
qcol = [c for c in df.columns if c.startswith("Q")][0]

print(df[[fcol, qcol]].to_string(index=True))
for i, row in df.iterrows():
    print(f"modo {i}: f = {row[fcol]:.5f} GHz,  Q = {row[qcol]:.3e}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.stem(df[fcol], df[qcol])
ax.set_xlabel("frecuencia del modo (GHz)"); ax.set_ylabel("Q"); ax.set_yscale("log")
ax.set_title("modos convergidos")
plt.tight_layout()


## Notas

- **Campos:** `eig_resonator/.../paraview/eigenmode/` tiene el campo E/B del/los
  modo(s) guardado(s) -- abri el `.pvd` en ParaView para ver la onda estacionaria.
- **`Q` interno:** aca lo fija la tangente de perdidas del sustrato
  (`LossTan` en el config) y las participaciones de interfaz. Sin puertos no
  hay `Q` de acoplamiento.
- **Eigenmode vs driven:** el eigenmode da el modo desnudo + `Q` interno barato
  y sin puerto; el driven da la respuesta medida (S21) con la carga de la
  linea. Usa eigenmode para saber donde barrer, driven para el S21.
